# AI-Powered SOC Analyst - Development Notebook
Run the cells top to bottom (open this notebook from inside `soc-ai-analyst/notebooks/`). Each section = one roadmap step.

In [ ]:
import sys, os
os.chdir('..')            # project root
sys.path.insert(0, os.getcwd())
import pandas as pd
pd.set_option('display.width', 200)
print(os.getcwd())

## Step 1 - Check environment

In [ ]:
import pandas, fastapi, streamlit, plotly, requests
from src.llm import ollama_available, OLLAMA_MODEL
print('pandas', pandas.__version__)
print('Ollama running:', ollama_available(), '| model:', OLLAMA_MODEL)

## Step 2 - Generate sample logs (with hidden attacks)

In [ ]:
from src.generator import write_samples
truth = write_samples()
truth

In [ ]:
print(open('data/sample_logs/auth.log').read()[:800])

## Step 3 - Parse logs into one common schema

In [ ]:
from src.parser import parse_file
linux = parse_file('data/sample_logs/auth.log')
win = parse_file('data/sample_logs/windows_security.csv')
print(linux.shape, win.shape)
linux.head()

In [ ]:
win.head()

In [ ]:
linux.groupby(['event_type','status']).size()

## Step 4 - Detection engine

In [ ]:
from src.detector import detect
alerts = detect(linux)
for a in alerts:
    print(f"{a['rule']:25} {a['source_ip']:16} count={a['count']:3}  {a['description']}")

## Step 5 - MITRE ATT&CK mapping

In [ ]:
from src.mitre import map_alerts
alerts = map_alerts(alerts)
pd.DataFrame([{'rule': a['rule'], 'ip': a['source_ip'], **a['mitre']} for a in alerts])

## Step 6 - Risk scoring

In [ ]:
from src.scoring import score_alerts
alerts = score_alerts(alerts)
pd.DataFrame([{'rule': a['rule'], 'ip': a['source_ip'], 'score': a['risk_score'], 'severity': a['severity']} for a in alerts])

## Step 7 - LLM explanation (Ollama, with template fallback)
Start Ollama first (`ollama serve`, `ollama pull llama3.2`) to get real LLM output.

In [ ]:
from src.llm import build_prompt, explain_alert
print(build_prompt(alerts[0])[:1200])

In [ ]:
from IPython.display import Markdown
ex = explain_alert(alerts[0])          # use_llm=False to force the template
print('source:', ex['source'], ex['warnings'])
Markdown(ex['text'])

## Step 8 - Full pipeline + SQLite storage

In [ ]:
from src.pipeline import analyze_file
from src import db
db.init_db()
df, alerts = analyze_file('data/sample_logs/auth.log')
upload_id, ids = db.save_analysis('auth.log', df, alerts)
print(db.stats())
pd.DataFrame(db.list_alerts(upload_id))

In [ ]:
# Generate an incident report for all alerts and save it
from src.llm import incident_report
exps = {a['alert_id']: explain_alert(a) for a in alerts}
report = incident_report(alerts, exps, title='SOC Incident Report - auth.log')
os.makedirs('reports', exist_ok=True)
open('reports/incident_report.md', 'w').write(report)
Markdown(report[:1500])

## Step 9 - Run the API and dashboard (in a terminal, not this notebook)
```
uvicorn api.main:app --reload        # API docs: http://127.0.0.1:8000/docs
streamlit run dashboard/app.py       # dashboard
```

## Step 10 - Run tests


In [ ]:
!python -m pytest -q tests